# Olist delivery risk

One row represents an order with a recorded delivery. The target is late when the actual
delivery falls on a calendar date after the promised date. Delivery on the promised date
counts as on time.

The data-quality and monthly summaries focus on development orders approved before June 1,
2018. Feature relationships use orders approved and delivered before March 1, 2018, so their
labels would have been known at the training cutoff.

## Load and audit the order data

In [1]:
from pathlib import Path

import pandas as pd

abt_path = Path("data/processed/logistic_abt.csv")
if Path.cwd().name == "notebooks":
    abt_path = Path("..") / abt_path

date_columns = [
    "order_approved_at",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(abt_path, parse_dates=date_columns)

In [2]:
order_ids = df["order_id"]
assert order_ids.notna().all(), "Every order must have an ID."
assert order_ids.is_unique, "The ABT must have one row per order ID."
assert df[date_columns].notna().all().all(), "Order timestamps must be present."
assert df["is_late"].isin([0, 1]).all(), "The late label must be binary."

promised_day = df["order_estimated_delivery_date"].dt.normalize()
actual_day = df["order_delivered_customer_date"].dt.normalize()
expected_late = actual_day > promised_day
label_mismatches = df["is_late"].ne(expected_late).sum()
assert label_mismatches == 0, "The late label must agree with the calendar-day rule."

audit_counts = {
    "Orders": len(df),
    "Unique order IDs": order_ids.nunique(),
    "Late-label mismatches": label_mismatches,
    "Orders delivered on their promised date": actual_day.eq(promised_day).sum(),
}
pd.Series(audit_counts, name="Count").to_frame()

,Count
Orders,96394
Unique order IDs,96394
Late-label mismatches,0
Orders delivered on their promised date,1291


## Order volume and late deliveries over time

Group orders by approval month and count their recorded outcomes. Read late rates alongside
order counts, especially for months with very few orders.

In [3]:
development_end = pd.Timestamp("2018-06-01")
training_cutoff = pd.Timestamp("2018-03-01")

approved_at = df["order_approved_at"]
delivered_at = df["order_delivered_customer_date"]
development = df.loc[approved_at < development_end]

approved_before_training = approved_at < training_cutoff
delivered_before_training = delivered_at < training_cutoff
training = df.loc[approved_before_training & delivered_before_training]

In [4]:
def summarize_lateness(orders, group):
    grouped_labels = orders.groupby(group, observed=True, dropna=False)["is_late"]
    summary = grouped_labels.agg(
        orders="size",
        late_orders="sum",
        late_rate="mean",
    )
    summary["late_rate_pct"] = (summary["late_rate"] * 100).round(2)
    return summary.drop(columns="late_rate")

In [5]:
approval_month = development["order_approved_at"].dt.to_period("M")
summarize_lateness(development, approval_month).rename_axis("Approval month")

,orders,late_orders,late_rate_pct
Approval month,,,
2016-09,1,1,100.00
2016-10,265,2,0.75
2016-12,1,0,0.00
2017-01,715,21,2.94
2017-02,1638,46,2.81
2017-03,2553,118,4.62
2017-04,2278,149,6.54
2017-05,3547,108,3.04
2017-06,3143,95,3.02


## Feature quality

Count missing values and unique observed values. Inspect numeric tails using percentiles and
maxima. The percentiles describe the observed distribution and help put the extremes in
context.

In [6]:
categorical_features = ["customer_city", "customer_state"]
numeric_features = [
    "days_from_approval_to_estimate",
    "item_count",
    "distinct_product_count",
    "distinct_seller_count",
    "price_sum",
    "freight_sum",
    "total_weight_g",
    "total_volume_cm3",
    "max_seller_distance_km",
]
features = development[categorical_features + numeric_features]

missing_count = features.isna().sum()
missing_pct = (missing_count / len(features) * 100).round(2)
feature_quality = pd.DataFrame({
    "Missing": missing_count,
    "Missing (%)": missing_pct,
    "Unique observed values": features.nunique(),
})
feature_quality.sort_values("Missing", ascending=False)

,Missing,Missing (%),Unique observed values
max_seller_distance_km,400,0.51,73137
total_weight_g,16,0.02,2583
total_volume_cm3,16,0.02,5555
days_from_approval_to_estimate,0,0.00,96
customer_state,0,0.00,27
customer_city,0,0.00,3878
item_count,0,0.00,17
price_sum,0,0.00,6509
distinct_seller_count,0,0.00,5
distinct_product_count,0,0.00,7


In [7]:
tail_percentiles = [0.01, 0.50, 0.95, 0.99]
numeric_summary = features[numeric_features].describe(percentiles=tail_percentiles)
numeric_summary.T.round(2)

,count,mean,std,min,1%,50%,95%,99%,max
days_from_approval_to_estimate,77784.0,24.55,8.16,1.00,9.83,24.00,38.00,51.00,154.00
item_count,77784.0,1.14,0.54,1.00,1.00,1.00,2.00,3.00,21.00
distinct_product_count,77784.0,1.04,0.22,1.00,1.00,1.00,1.00,2.00,7.00
distinct_seller_count,77784.0,1.01,0.12,1.00,1.00,1.00,1.00,2.00,5.00
price_sum,77784.0,136.84,206.40,0.85,11.99,86.90,399.00,990.00,13440.00
freight_sum,77784.0,22.29,20.33,0.00,7.39,16.79,52.86,101.17,1002.29
total_weight_g,77768.0,2464.01,4908.62,2.00,100.00,800.00,10750.00,22608.25,184400.00
total_volume_cm3,77768.0,17955.10,31115.76,288.00,352.00,7632.00,65988.00,139748.75,1476000.00
max_seller_distance_km,77384.0,609.47,593.08,0.00,5.96,444.04,2100.69,2485.79,4946.13


## Lateness within the training cohort

Compare late rates across five quantile bands for the promise window and seller distance,
and across customer states. Ties at quantile boundaries can produce bands of different
sizes. Keep missing distances visible as their own group.

In [8]:
training_labels = training["is_late"]
training_counts = {
    "Orders": len(training),
    "Late orders": training_labels.sum(),
    "Late rate (%)": round(training_labels.mean() * 100, 2),
}
pd.Series(training_counts, name="Training cohort").to_frame()

,Training cohort
Orders,53618.00
Late orders,2710.00
Late rate (%),5.05


In [9]:
promise_days = training["days_from_approval_to_estimate"]
promise_bands = pd.qcut(promise_days, q=5, duplicates="drop")
summarize_lateness(training, promise_bands).rename_axis("Promise window (days)")

,orders,late_orders,late_rate_pct
Promise window (days),,,
"(4.999, 19.0]",11221,548,4.88
"(19.0, 23.0]",13191,710,5.38
"(23.0, 26.0]",9182,481,5.24
"(26.0, 31.0]",10611,588,5.54
"(31.0, 154.0]",9413,383,4.07


In [10]:
seller_distance = training["max_seller_distance_km"]
distance_bands = pd.qcut(seller_distance, q=5, duplicates="drop")
distance_bands = distance_bands.cat.add_categories(["Missing"]).fillna("Missing")

distance_rates = summarize_lateness(training, distance_bands)
distance_rates.rename_axis("Maximum seller distance (km)")

,orders,late_orders,late_rate_pct
Maximum seller distance (km),,,
"(-0.001, 131.68]",10669,382,3.58
"(131.68, 354.113]",10668,499,4.68
"(354.113, 542.714]",10668,489,4.58
"(542.714, 883.92]",10668,562,5.27
"(883.92, 4946.133]",10668,761,7.13
Missing,277,17,6.14


In [11]:
state_rates = summarize_lateness(training, "customer_state")
state_rates.sort_values("orders", ascending=False)

,orders,late_orders,late_rate_pct
customer_state,,,
SP,21734,748,3.44
RJ,7115,682,9.59
MG,6506,160,2.46
RS,3049,145,4.76
PR,2740,74,2.70
SC,2036,144,7.07
BA,1804,162,8.98
ES,1147,78,6.80
GO,1121,53,4.73


## Findings

- Lateness varied sharply across months with thousands of orders. March 2018 had a 19.20% late
  rate, compared with 4.62% in April and 6.54% in May.
- Maximum seller distance was missing for 400 development orders (0.51%). Weight and volume
  were each missing for 16. The maximum order price was R$13,440, compared with R$990 at the 99th
  percentile. The longest promise window was 154 days, compared with 51 days at the 99th percentile.
- In the training cohort, 3.58% of orders in the nearest distance band arrived late, compared
  with 7.13% in the farthest band. Late rates showed no steady trend across promise-window bands.
  Among high-volume states, Rio de Janeiro had a 9.59% late rate, compared with 3.44% in São Paulo
  and 2.46% in Minas Gerais.

## Constant-probability baseline

The dummy classifier assigns every order the late rate observed in training.
Evaluate it separately on March and April approvals whose deliveries were
recorded before June 1. With tied scores, average precision equals the
late rate in each month and ROC-AUC is 0.5.

In [12]:
validation_end = pd.Timestamp("2018-05-01")

approved_in_validation = (approved_at >= training_cutoff) & (approved_at < validation_end)
delivered_before_evaluation = delivered_at < development_end
validation = df.loc[approved_in_validation & delivered_before_evaluation]

In [13]:
import numpy as np
from sklearn.dummy import DummyClassifier

assert training_labels.nunique() == 2, "Training requires both outcome classes."

# Placeholder inputs give the dummy classifier the row counts.
train_placeholder = np.zeros((len(training), 1))
validation_placeholder = np.zeros((len(validation), 1))

dummy = DummyClassifier(strategy="prior")
dummy.fit(train_placeholder, training_labels)

late_class_index = dummy.classes_.tolist().index(1)
late_probabilities = dummy.predict_proba(validation_placeholder)[:, late_class_index]
training_late_rate = training_labels.mean()
assert np.allclose(late_probabilities, training_late_rate)

In [14]:
from sklearn.metrics import average_precision_score, roc_auc_score

def score_by_month(outcomes, probabilities):
    scored_orders = outcomes.assign(late_probability=probabilities)
    results = []
    for month, orders in scored_orders.groupby("approval_month"):
        labels = orders["is_late"]
        scores = orders["late_probability"]
        assert labels.nunique() == 2, f"{month} requires both outcome classes."

        results.append({
            "approval_month": month,
            "orders": len(orders),
            "late_orders": labels.sum(),
            "late_rate": labels.mean(),
            "average_precision": average_precision_score(labels, scores),
            "roc_auc": roc_auc_score(labels, scores),
        })

    return pd.DataFrame(results).set_index("approval_month")

In [15]:
validation_outcomes = validation[["is_late"]].assign(
    approval_month=validation["order_approved_at"].dt.to_period("M"),
)

baseline_metrics = score_by_month(validation_outcomes, late_probabilities)
baseline_metrics.insert(3, "predicted_late_probability", training_late_rate)
assert np.allclose(baseline_metrics["average_precision"], baseline_metrics["late_rate"])
assert np.allclose(baseline_metrics["roc_auc"], 0.5)

baseline_metrics.round(4)

,orders,late_orders,late_rate,predicted_late_probability,average_precision,roc_auc
approval_month,,,,,,
2018-03,7076,1353,0.1912,0.0505,0.1912,0.5
2018-04,6549,238,0.0363,0.0505,0.0363,0.5


## Training feature diagnostics

Inspect Spearman correlations between numeric predictors to find overlapping signals.
Estimate mutual information with the late label to check for associations of any shape.
Both tables use the training cohort. Mutual information treats state, counts, and the
number of promised days as discrete. These estimates describe individual features and
can be interpreted alongside the validation results.

In [16]:
numeric_training = training[numeric_features]
spearman_correlation = numeric_training.corr(method="spearman")
spearman_correlation.round(2)

,days_from_approval_to_estimate,item_count,distinct_product_count,distinct_seller_count,price_sum,freight_sum,total_weight_g,total_volume_cm3,max_seller_distance_km
days_from_approval_to_estimate,1.00,0.00,0.00,0.01,0.10,0.49,0.06,0.05,0.65
item_count,0.00,1.00,0.56,0.32,0.18,0.40,0.19,0.19,0.01
distinct_product_count,0.00,0.56,1.00,0.57,0.11,0.22,0.09,0.08,0.01
distinct_seller_count,0.01,0.32,0.57,1.00,0.07,0.13,0.07,0.07,0.03
price_sum,0.10,0.18,0.11,0.07,1.00,0.49,0.56,0.44,0.09
freight_sum,0.49,0.40,0.22,0.13,0.49,1.00,0.53,0.44,0.59
total_weight_g,0.06,0.19,0.09,0.07,0.56,0.53,1.00,0.78,0.03
total_volume_cm3,0.05,0.19,0.08,0.07,0.44,0.44,0.78,1.00,0.01
max_seller_distance_km,0.65,0.01,0.01,0.03,0.09,0.59,0.03,0.01,1.00


In [17]:
from sklearn.feature_selection import mutual_info_classif

mi_input = numeric_training.fillna(numeric_training.median())
state_codes, _ = pd.factorize(training["customer_state"].fillna("Missing"))
mi_input = mi_input.assign(customer_state=state_codes)

discrete_predictors = [
    "days_from_approval_to_estimate",
    "item_count",
    "distinct_product_count",
    "distinct_seller_count",
    "customer_state",
]
mi_scores = mutual_info_classif(
    mi_input,
    training_labels,
    discrete_features=mi_input.columns.isin(discrete_predictors),
    random_state=42,
)

mutual_information = pd.Series(
    mi_scores, index=mi_input.columns, name="Mutual information (nats)"
)
mutual_information.sort_values(ascending=False).round(5).to_frame()

,Mutual information (nats)
customer_state,0.00746
freight_sum,0.00413
total_volume_cm3,0.00395
price_sum,0.00263
total_weight_g,0.00232
days_from_approval_to_estimate,0.00155
max_seller_distance_km,0.00091
distinct_seller_count,0.00027
item_count,0.00016
distinct_product_count,0.00011


## Logistic regression

Compare the promised delivery window alone with the full order features.
For each feature set, try two regularization strengths (C = 0.1 and C = 1).
Each pipeline learns numeric imputation, scaling, and any state encoding
during fitting on the training rows.

In [18]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

promise_features = ["days_from_approval_to_estimate"]
full_features = numeric_features + ["customer_state"]

numeric_preprocessing = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
full_preprocessing = ColumnTransformer([
    ("numeric", numeric_preprocessing, numeric_features),
    ("state", OneHotEncoder(handle_unknown="ignore"), ["customer_state"]),
])

logistic_inputs = {
    "Promise window": (promise_features, numeric_preprocessing),
    "Full order": (full_features, full_preprocessing),
}

In [19]:
logistic_probabilities = {}
for feature_set, (columns, preprocessing) in logistic_inputs.items():
    train_features = training[columns]
    validation_features = validation[columns]

    for c in (0.1, 1.0):
        classifier = LogisticRegression(C=c, max_iter=500)
        pipeline = make_pipeline(clone(preprocessing), classifier)
        pipeline.fit(train_features, training_labels)

        late_index = pipeline.classes_.tolist().index(1)
        probabilities = pipeline.predict_proba(validation_features)[:, late_index]
        logistic_probabilities[(feature_set, c)] = probabilities

In [20]:
monthly_results = []
for (feature_set, c), probabilities in logistic_probabilities.items():
    scores = score_by_month(validation_outcomes, probabilities)
    scores["ap_vs_dummy"] = scores["average_precision"] - baseline_metrics["average_precision"]
    monthly_results.append(scores.assign(feature_set=feature_set, C=c).reset_index())

logistic_metrics = pd.concat(monthly_results, ignore_index=True)
logistic_metrics = logistic_metrics.set_index(["feature_set", "C", "approval_month"])
report_columns = ["orders", "average_precision", "roc_auc", "ap_vs_dummy"]
logistic_metrics[report_columns].round(4)

orders  ...  ap_vs_dummy
feature_set    C   approval_month          ...             
Promise window 0.1 2018-03           7076  ...      -0.0246
                   2018-04           6549  ...       0.0044
               1.0 2018-03           7076  ...      -0.0246
                   2018-04           6549  ...       0.0044
Full order     0.1 2018-03           7076  ...       0.1485
                   2018-04           6549  ...       0.1225
               1.0 2018-03           7076  ...       0.1446
                   2018-04           6549  ...       0.1207

[8 rows x 4 columns]

In [21]:
mean_logistic_ap = logistic_metrics.groupby(["feature_set", "C"])["average_precision"].mean()
mean_logistic_ap = mean_logistic_ap.sort_values(ascending=False).rename("Mean monthly AP")
mean_logistic_ap.round(4).to_frame()

Mean monthly AP
feature_set    C                   
Full order     0.1           0.2493
               1.0           0.2464
Promise window 0.1           0.1037
               1.0           0.1037

### Logistic regression findings

- Weight and volume were strongly correlated in training (Spearman 0.78).
- The promise-window model fell below the dummy on March average precision and
  improved on it slightly in April.
- Both full-order models beat the dummy in both months. The full-order model with
  C = 0.1 had the highest mean monthly average precision among the logistic fits.

## Decision tree

Fit depths 3 and 6 on both feature sets with at least 100 training orders per leaf.
Fit median imputation and state encoding on the same training orders used for
logistic regression. Evaluate each tree on the same March and April orders.

In [22]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree_numeric_preprocessing = SimpleImputer(strategy="median")
tree_full_preprocessing = ColumnTransformer(
    [
        ("numeric", tree_numeric_preprocessing, numeric_features),
        ("state", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["customer_state"]),
    ],
    verbose_feature_names_out=False,
)

tree_inputs = {
    "Promise window": (promise_features, tree_numeric_preprocessing),
    "Full order": (full_features, tree_full_preprocessing),
}

In [23]:
tree_models = {}
for feature_set, (columns, preprocessing) in tree_inputs.items():
    train_features = training[columns]

    for depth in (3, 6):
        classifier = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=100,
            random_state=42,
        )
        pipeline = make_pipeline(clone(preprocessing), classifier)
        tree_models[(feature_set, depth)] = pipeline.fit(train_features, training_labels)

In [24]:
tree_probabilities = {}
tree_monthly_results = []
for (feature_set, depth), pipeline in tree_models.items():
    columns, _ = tree_inputs[feature_set]
    late_index = pipeline.classes_.tolist().index(1)
    probabilities = pipeline.predict_proba(validation[columns])[:, late_index]
    tree_probabilities[(feature_set, depth)] = probabilities

    scores = score_by_month(validation_outcomes, probabilities)
    scores["ap_vs_dummy"] = scores["average_precision"] - baseline_metrics["average_precision"]
    tree_monthly_results.append(
        scores.assign(feature_set=feature_set, max_depth=depth).reset_index()
    )

tree_metrics = pd.concat(tree_monthly_results, ignore_index=True)
tree_metrics = tree_metrics.set_index(["feature_set", "max_depth", "approval_month"])
tree_metrics[report_columns].round(4)

orders  ...  ap_vs_dummy
feature_set    max_depth approval_month          ...             
Promise window 3         2018-03           7076  ...       0.0280
                         2018-04           6549  ...       0.0018
               6         2018-03           7076  ...       0.0302
                         2018-04           6549  ...       0.0012
Full order     3         2018-03           7076  ...       0.1072
                         2018-04           6549  ...       0.0579
               6         2018-03           7076  ...       0.1039
                         2018-04           6549  ...       0.0959

[8 rows x 4 columns]

In [25]:
tree_mean_ap = tree_metrics.groupby(["feature_set", "max_depth"], sort=False)["average_precision"].mean()
tree_mean_ap = tree_mean_ap.sort_values(ascending=False, kind="stable").rename("Mean monthly AP")
tree_mean_ap.round(4).to_frame()

Mean monthly AP
feature_set    max_depth                 
Full order     6                   0.2137
               3                   0.1963
Promise window 6                   0.1295
               3                   0.1287

In [26]:
best_tree = tree_models[tree_mean_ap.index[0]]
best_tree_features = best_tree[0].get_feature_names_out()
first_tree_splits = export_text(
    best_tree[-1],
    feature_names=best_tree_features.tolist(),
    max_depth=1,
)
print(first_tree_splits)

|--- customer_state_RJ <= 0.50
|   |--- max_seller_distance_km <= 1340.87
|   |   |--- truncated branch of depth 5
|   |--- max_seller_distance_km >  1340.87
|   |   |--- truncated branch of depth 5
|--- customer_state_RJ >  0.50
|   |--- max_seller_distance_km <= 285.87
|   |   |--- truncated branch of depth 4
|   |--- max_seller_distance_km >  285.87
|   |   |--- truncated branch of depth 5



### Decision tree findings

- The full-order tree at depth 6 had the highest mean monthly average precision of the
  four trees (0.2137). The leading logistic regression still scored higher in both months.
- The leading tree first split on whether the customer was in Rio de Janeiro. Its next
  split used seller distance, with a cutoff of about 286 km for Rio de Janeiro and
  1,341 km for other states.

## Random forest

Fit one 100-tree forest per feature set. Each tree uses the same depth and leaf
limits as the depth-6 decision tree. The forest trains trees on bootstrap samples
and considers a random subset of features at each split. Fit preprocessing on the
shared training cohort and compare average precision on the same validation orders.

In [27]:
from sklearn.ensemble import RandomForestClassifier

forest_models = {}
for feature_set, (columns, preprocessing) in tree_inputs.items():
    classifier = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        min_samples_leaf=100,
        max_features="sqrt",
        random_state=42,
        n_jobs=2,
    )
    pipeline = make_pipeline(clone(preprocessing), classifier)
    forest_models[feature_set] = pipeline.fit(training[columns], training_labels)

In [28]:
forest_probabilities = {}
forest_monthly_results = []
for feature_set, pipeline in forest_models.items():
    columns, _ = tree_inputs[feature_set]
    late_index = pipeline.classes_.tolist().index(1)
    probabilities = pipeline.predict_proba(validation[columns])[:, late_index]
    forest_probabilities[feature_set] = probabilities

    scores = score_by_month(validation_outcomes, probabilities)
    forest_monthly_results.append(scores.assign(feature_set=feature_set).reset_index())

forest_metrics = pd.concat(forest_monthly_results, ignore_index=True)
forest_metrics = forest_metrics.set_index(["feature_set", "approval_month"])

depth_6_scores = tree_metrics.xs(6, level="max_depth")
forest_metrics["depth_6_tree_ap"] = depth_6_scores["average_precision"]
forest_metrics["ap_vs_depth_6_tree"] = (
    forest_metrics["average_precision"] - forest_metrics["depth_6_tree_ap"]
)

forest_report_columns = [
    "orders", "average_precision", "roc_auc", "depth_6_tree_ap", "ap_vs_depth_6_tree"
]
forest_metrics[forest_report_columns].round(4)

orders  ...  ap_vs_depth_6_tree
feature_set    approval_month          ...                    
Promise window 2018-03           7076  ...             -0.0017
               2018-04           6549  ...             -0.0011
Full order     2018-03           7076  ...              0.0477
               2018-04           6549  ...             -0.0087

[4 rows x 5 columns]

In [29]:
forest_mean_ap = forest_metrics.groupby("feature_set", sort=False)["average_precision"].mean()
forest_mean_ap = forest_mean_ap.sort_values(ascending=False).rename("Mean monthly AP")
forest_mean_ap.round(4).to_frame()

,Mean monthly AP
feature_set,
Full order,0.2331
Promise window,0.1281


### Random forest findings

The full-order forest gained 0.0477 AP over the depth-6 tree in March and lost
0.0087 AP in April. ROC-AUC improved in both months. Its mean AP of 0.2331 was
below the leading logistic regression's 0.2493.

## Histogram gradient boosting

Compare 15 and 31 maximum leaves on both feature sets. Fit 100 boosting iterations
with a learning rate of 0.1 for every candidate. Use the same training-fitted
preprocessing as the other tree models and evaluate on the same March and April orders.

In [30]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting_models = {}
for feature_set, (columns, preprocessing) in tree_inputs.items():
    train_features = training[columns]

    for leaf_limit in (15, 31):
        classifier = HistGradientBoostingClassifier(
            max_leaf_nodes=leaf_limit,
            learning_rate=0.1,
            max_iter=100,
            early_stopping=False,
            random_state=42,
        )
        pipeline = make_pipeline(clone(preprocessing), classifier)
        boosting_models[(feature_set, leaf_limit)] = pipeline.fit(train_features, training_labels)

In [31]:
boosting_probabilities = {}
boosting_monthly_results = []

for (feature_set, leaf_limit), pipeline in boosting_models.items():
    columns, _ = tree_inputs[feature_set]
    late_index = pipeline.classes_.tolist().index(1)
    probabilities = pipeline.predict_proba(validation[columns])[:, late_index]
    boosting_probabilities[(feature_set, leaf_limit)] = probabilities

    scores = score_by_month(validation_outcomes, probabilities)
    scores["ap_vs_dummy"] = scores["average_precision"] - baseline_metrics["average_precision"]
    boosting_monthly_results.append(
        scores.assign(feature_set=feature_set, max_leaf_nodes=leaf_limit).reset_index()
    )

boosting_metrics = pd.concat(boosting_monthly_results, ignore_index=True)
boosting_metrics = boosting_metrics.set_index(["feature_set", "max_leaf_nodes", "approval_month"])
boosting_metrics[report_columns].round(4)

orders  ...  ap_vs_dummy
feature_set    max_leaf_nodes approval_month          ...             
Promise window 15             2018-03           7076  ...       0.0278
                              2018-04           6549  ...       0.0031
               31             2018-03           7076  ...       0.0278
                              2018-04           6549  ...       0.0031
Full order     15             2018-03           7076  ...       0.1462
                              2018-04           6549  ...       0.1254
               31             2018-03           7076  ...       0.1326
                              2018-04           6549  ...       0.1108

[8 rows x 4 columns]

In [32]:
boosting_ap = boosting_metrics.groupby(
    ["feature_set", "max_leaf_nodes"], sort=False
)["average_precision"].mean()

boosting_mean_ap = boosting_ap.sort_values(ascending=False, kind="stable")
boosting_mean_ap.rename("Mean monthly AP").round(4).to_frame()

Mean monthly AP
feature_set    max_leaf_nodes                 
Full order     15                       0.2496
               31                       0.2355
Promise window 31                       0.1293
               15                       0.1292

## Late deliveries found in the highest-risk 10%

Review the highest-scored 10% of orders in each validation month, rounding up to a
whole order and using order ID to resolve tied scores. Compare the leading configuration
from each model family. Precision is the late share of the review queue. Recall is
the share of the month's late deliveries that the queue captures.

In [33]:
from math import ceil

def score_review_queue(outcomes, probabilities, review_fraction):
    ranked_orders = outcomes.assign(late_probability=probabilities)
    results = []

    for month, orders in ranked_orders.groupby("approval_month"):
        review_count = ceil(len(orders) * review_fraction)
        ranked_month = orders.sort_values(
            ["late_probability", "order_id"], ascending=[False, True]
        )
        reviewed_orders = ranked_month.head(review_count)
        late_found = reviewed_orders["is_late"].sum()
        total_late = orders["is_late"].sum()

        results.append({
            "approval_month": month,
            "reviewed_orders": review_count,
            "late_found": late_found,
            "total_late": total_late,
            "precision": late_found / review_count,
            "recall": late_found / total_late,
        })

    return pd.DataFrame(results).set_index("approval_month")

In [34]:
priority_probabilities = {
    "Logistic regression": logistic_probabilities[mean_logistic_ap.index[0]],
    "Decision tree": tree_probabilities[tree_mean_ap.index[0]],
    "Random forest": forest_probabilities[forest_mean_ap.index[0]],
    "Gradient boosting": boosting_probabilities[boosting_mean_ap.index[0]],
}

In [35]:
review_fraction = 0.10
review_outcomes = validation_outcomes.assign(order_id=validation["order_id"])
review_results = []

for model_name, probabilities in priority_probabilities.items():
    results = score_review_queue(review_outcomes, probabilities, review_fraction)
    review_results.append(results.assign(model=model_name).reset_index())

review_metrics = pd.concat(review_results, ignore_index=True)
review_metrics = review_metrics.set_index(["model", "approval_month"])
review_metrics.round(4)

reviewed_orders  ...  recall
model               approval_month                   ...        
Logistic regression 2018-03                     708  ...  0.2306
                    2018-04                     655  ...  0.4748
Decision tree       2018-03                     708  ...  0.2106
                    2018-04                     655  ...  0.3782
Random forest       2018-03                     708  ...  0.2380
                    2018-04                     655  ...  0.3908
Gradient boosting   2018-03                     708  ...  0.2299
                    2018-04                     655  ...  0.4412

[8 rows x 5 columns]

### Gradient boosting and review-queue findings

Full-order boosting with 15 maximum leaves achieved a mean validation AP of 0.24960,
close to logistic regression at 0.24927. At the 10% review budget, boosting found
311 March and 105 April late orders. Logistic regression found 312 and 113.
Randomly reviewing the same number of orders would find about 135 and 24, respectively.

## Fixed probability cutoffs

Flag orders with late scores above each cutoff. Use the same cutoff in March and
April, and compare how many orders it flags, how many flagged orders arrive late,
and what share of all late orders it captures. Precision is undefined when the
cutoff flags zero orders.

Evaluate all 14 configurations and the dummy. The two monthly tables show the
dummy and the leading configuration from each family by mean average precision.

In [36]:
probability_cutoffs = [0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30, 0.50]

cutoff_candidates = {
    "Dummy": {("No features", "Prior"): late_probabilities},
    "Logistic regression": logistic_probabilities,
    "Decision tree": tree_probabilities,
    "Random forest": {
        (feature_set, 6): probabilities
        for feature_set, probabilities in forest_probabilities.items()
    },
    "Gradient boosting": boosting_probabilities,
}
leading_cutoff_candidates = {
    "Dummy": ("No features", "Prior"),
    "Logistic regression": mean_logistic_ap.index[0],
    "Decision tree": tree_mean_ap.index[0],
    "Random forest": (forest_mean_ap.index[0], 6),
    "Gradient boosting": boosting_mean_ap.index[0],
}

In [37]:
def score_cutoffs(outcomes, probabilities, cutoffs):
    scored_orders = outcomes.assign(late_probability=probabilities)
    results = []

    for month, orders in scored_orders.groupby("approval_month"):
        labels = orders["is_late"].to_numpy(dtype=bool)
        scores = orders["late_probability"].to_numpy()
        total_late = labels.sum()

        for cutoff in cutoffs:
            flagged = scores > cutoff
            flagged_count = flagged.sum()
            late_found = (flagged & labels).sum()

            results.append({
                "approval_month": month,
                "cutoff": cutoff,
                "orders": len(orders),
                "total_late": total_late,
                "flagged_orders": flagged_count,
                "late_found": late_found,
                "review_rate": flagged_count / len(orders),
                "precision": late_found / flagged_count if flagged_count else np.nan,
                "recall": late_found / total_late,
            })

    return pd.DataFrame(results)

In [38]:
cutoff_results = []
for model_name, candidates in cutoff_candidates.items():
    for candidate, probabilities in candidates.items():
        feature_set, setting = candidate
        scores = score_cutoffs(validation_outcomes, probabilities, probability_cutoffs)
        cutoff_results.append(scores.assign(
            model=model_name,
            feature_set=feature_set,
            setting=setting,
            is_leading_candidate=candidate == leading_cutoff_candidates[model_name],
        ))

cutoff_metrics = pd.concat(cutoff_results, ignore_index=True)

In [39]:
cutoff_report_columns = ["flagged_orders", "late_found", "review_rate", "precision", "recall"]
leading_cutoffs = cutoff_metrics.loc[cutoff_metrics["is_leading_candidate"]]
leading_cutoffs = leading_cutoffs.set_index(["approval_month", "model", "cutoff"])

march_cutoffs = leading_cutoffs.xs(pd.Period("2018-03", freq="M"))
march_cutoffs[cutoff_report_columns].round(4)

flagged_orders  late_found  ...  precision  recall
model               cutoff                              ...                   
Dummy               0.025             7076        1353  ...     0.1912  1.0000
                    0.050             7076        1353  ...     0.1912  1.0000
                    0.075                0           0  ...        NaN  0.0000
                    0.100                0           0  ...        NaN  0.0000
                    0.150                0           0  ...        NaN  0.0000
                    0.200                0           0  ...        NaN  0.0000
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Logistic regression 0.025             6440        1315  ...     0.2042  0.9719
                    0.050             2987         838  ...     0.2805  0.6194
                    0.075             1456         567  ...     0.3894  0.4191
                    0.100              869         392  ...     0.4511  0.2897
                    0.150              162          67  ...     0.4136  0.0495
                    0.200               45          22  ...     0.4889  0.0163
                    0.300                6           3  ...     0.5000  0.0022
                    0.500                1           1  ...     1.0000  0.0007
Decision tree       0.025             6896        1339  ...     0.1942  0.9897
                    0.050             2665         716  ...     0.2687  0.5292
                    0.075             1446         519  ...     0.3589  0.3836
                    0.100              936         349  ...     0.3729  0.2579
                    0.150              293         117  ...     0.3993  0.0865
                    0.200              175          83  ...     0.4743  0.0613
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Random forest       0.025             7076        1353  ...     0.1912  1.0000
                    0.050             3061         863  ...     0.2819  0.6378
                    0.075             1151         473  ...     0.4109  0.3496
                    0.100              175          86  ...     0.4914  0.0636
                    0.150                0           0  ...        NaN  0.0000
                    0.200                0           0  ...        NaN  0.0000
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Gradient boosting   0.025             6077        1264  ...     0.2080  0.9342
                    0.050             3207         900  ...     0.2806  0.6652
                    0.075             1724         620  ...     0.3596  0.4582
                    0.100              960         396  ...     0.4125  0.2927
                    0.150              314         135  ...     0.4299  0.0998
                    0.200              111          47  ...     0.4234  0.0347
                    0.300               22          10  ...     0.4545  0.0074
                    0.500                0           0  ...        NaN  0.0000

[40 rows x 5 columns]

In [40]:
april_cutoffs = leading_cutoffs.xs(pd.Period("2018-04", freq="M"))
april_cutoffs[cutoff_report_columns].round(4)

flagged_orders  late_found  ...  precision  recall
model               cutoff                              ...                   
Dummy               0.025             6549         238  ...     0.0363  1.0000
                    0.050             6549         238  ...     0.0363  1.0000
                    0.075                0           0  ...        NaN  0.0000
                    0.100                0           0  ...        NaN  0.0000
                    0.150                0           0  ...        NaN  0.0000
                    0.200                0           0  ...        NaN  0.0000
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Logistic regression 0.025             5715         236  ...     0.0413  0.9916
                    0.050             2195         176  ...     0.0802  0.7395
                    0.075             1050         136  ...     0.1295  0.5714
                    0.100              566         108  ...     0.1908  0.4538
                    0.150              121          27  ...     0.2231  0.1134
                    0.200               38          10  ...     0.2632  0.0420
                    0.300                5           2  ...     0.4000  0.0084
                    0.500                0           0  ...        NaN  0.0000
Decision tree       0.025             6298         236  ...     0.0375  0.9916
                    0.050             2090         146  ...     0.0699  0.6134
                    0.075             1072         110  ...     0.1026  0.4622
                    0.100              600          84  ...     0.1400  0.3529
                    0.150              229          47  ...     0.2052  0.1975
                    0.200              106          32  ...     0.3019  0.1345
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Random forest       0.025             6549         238  ...     0.0363  1.0000
                    0.050             2703         172  ...     0.0636  0.7227
                    0.075              851         106  ...     0.1246  0.4454
                    0.100              136          30  ...     0.2206  0.1261
                    0.150                0           0  ...        NaN  0.0000
                    0.200                0           0  ...        NaN  0.0000
                    0.300                0           0  ...        NaN  0.0000
                    0.500                0           0  ...        NaN  0.0000
Gradient boosting   0.025             5023         222  ...     0.0442  0.9328
                    0.050             2312         179  ...     0.0774  0.7521
                    0.075             1235         135  ...     0.1093  0.5672
                    0.100              689         107  ...     0.1553  0.4496
                    0.150              216          57  ...     0.2639  0.2395
                    0.200               76          23  ...     0.3026  0.0966
                    0.300               17           4  ...     0.2353  0.0168
                    0.500                0           0  ...        NaN  0.0000

[40 rows x 5 columns]

### Cutoff findings

For full-order logistic regression, a 0.10 cutoff flagged 12.3% of March orders
and 8.6% of April orders. Recall was 29.0% and 45.4%. Lowering the cutoff to 0.05
raised recall to 61.9% and 73.9%, with more than a third of orders flagged in each
month. At 0.10, the forest flagged about 2% in both months, so a probability
cutoff implies different review workloads across models.

## Consolidated validation and model selection

Compare all 14 configurations on the same March and April orders. The table is
sorted by mean monthly average precision. Late orders found are measured at a
10% review budget per month. The dummy row shows the expected number found by
randomly reviewing the same number of orders.

In [41]:
configuration_labels = {
    "Dummy": "{}",
    "Logistic regression": "C={:g}",
    "Decision tree": "depth={}",
    "Random forest": "100 trees, depth={}",
    "Gradient boosting": "max leaves={}",
}

candidate_results = []
for model_name, candidates in cutoff_candidates.items():
    for (feature_set, setting), probabilities in candidates.items():
        ranking_scores = score_by_month(validation_outcomes, probabilities)
        queue_scores = score_review_queue(review_outcomes, probabilities, review_fraction)

        if model_name == "Dummy":
            expected_late = queue_scores["reviewed_orders"] * ranking_scores["late_rate"]
            queue_scores = queue_scores.assign(late_found=expected_late)

        configuration = configuration_labels[model_name].format(setting)
        candidate_scores = ranking_scores.join(queue_scores[["late_found"]])
        candidate_results.append(candidate_scores.reset_index().assign(
            model=model_name,
            feature_set=feature_set,
            configuration=configuration,
        ))

candidate_index = ["model", "feature_set", "configuration"]
validation_by_candidate = pd.concat(candidate_results, ignore_index=True)
validation_by_candidate = validation_by_candidate.set_index(candidate_index + ["approval_month"])

In [42]:
selected_feature_set = "Full order"
selected_c = 0.1
selected_configuration = ("Logistic regression", selected_feature_set, f"C={selected_c:g}")

metric_labels = {
    "average_precision": "AP",
    "roc_auc": "ROC-AUC",
    "late_found": "Late found",
}
validation_summary = validation_by_candidate[list(metric_labels)].unstack("approval_month")
validation_summary.columns = [
    f"{metric_labels[metric]} {month:%b}" for metric, month in validation_summary.columns
]
validation_summary["Mean AP"] = validation_summary[["AP Mar", "AP Apr"]].mean(axis=1)
validation_summary["Total late found"] = validation_summary["Late found Mar"] + validation_summary["Late found Apr"]
validation_summary["Selected"] = validation_summary.index == selected_configuration
validation_summary = validation_summary.sort_values("Mean AP", ascending=False, kind="stable")

summary_columns = [
    "Mean AP", "AP Mar", "AP Apr", "ROC-AUC Mar", "ROC-AUC Apr",
    "Late found Mar", "Late found Apr", "Total late found", "Selected",
]
validation_summary[summary_columns].round(5)

Mean AP  ...  Selected
model               feature_set    configuration                ...          
Gradient boosting   Full order     max leaves=15       0.24960  ...     False
Logistic regression Full order     C=0.1               0.24927  ...      True
                                   C=1                 0.24642  ...     False
Gradient boosting   Full order     max leaves=31       0.23546  ...     False
Random forest       Full order     100 trees, depth=6  0.23313  ...     False
Decision tree       Full order     depth=6             0.21367  ...     False
                                   depth=3             0.19630  ...     False
                    Promise window depth=6             0.12948  ...     False
Gradient boosting   Promise window max leaves=31       0.12926  ...     False
                                   max leaves=15       0.12925  ...     False
Decision tree       Promise window depth=3             0.12869  ...     False
Random forest       Promise window 100 trees, depth=6  0.12807  ...     False
Dummy               No features    Prior               0.11378  ...     False
Logistic regression Promise window C=0.1               0.10370  ...     False
                                   C=1                 0.10370  ...     False

[15 rows x 9 columns]

### Selected model and review rule

Select full-order logistic regression with C = 0.1 for Step 5. At the
10% review budget, it found 425 late orders across March and April.
Boosting with 15 maximum leaves led mean AP (0.24960) and found 416
late orders. Logistic regression with C = 1 found 426 and had mean AP
of 0.24642. The C = 0.1 fit combined nearly the same review yield with
higher mean AP (0.24927) and stronger regularization.

For each holdout month, rank by the predicted late score and review
the highest-scored 10%, rounding up and resolving ties by order ID.
Apply this rule in Step 5 after refitting on eligible pre-June orders.

## Final holdout evaluation

Refit the selected full-order logistic regression with C = 0.1 using orders
approved and delivered before June 1, 2018. Score June through August approvals
once and reuse those scores for ranking metrics and the 10% monthly review queue.
This evaluation represents orders with recorded deliveries.

In [43]:
holdout_end = pd.Timestamp("2018-09-01")
refit_orders = development.loc[development["order_delivered_customer_date"] < development_end]
holdout_orders = df.loc[(approved_at >= development_end) & (approved_at < holdout_end)]

assert refit_orders["order_approved_at"].lt(development_end).all()
assert refit_orders["order_delivered_customer_date"].lt(development_end).all()
assert refit_orders.index.intersection(holdout_orders.index).empty

cohort_sizes = {"Refit": len(refit_orders), "Holdout": len(holdout_orders)}
pd.Series(cohort_sizes, name="Orders").to_frame()

,Orders
Refit,75388
Holdout,18610


In [44]:
selected_columns, selected_preprocessing = logistic_inputs[selected_feature_set]
refit_labels = refit_orders["is_late"]

final_classifier = LogisticRegression(C=selected_c, max_iter=500)
final_model = make_pipeline(clone(selected_preprocessing), final_classifier)
final_model.fit(refit_orders[selected_columns], refit_labels)

final_late_index = final_model.classes_.tolist().index(1)
holdout_probabilities = final_model.predict_proba(
    holdout_orders[selected_columns]
)[:, final_late_index]

In [45]:
holdout_outcomes = holdout_orders[["order_id", "is_late"]].assign(
    approval_month=holdout_orders["order_approved_at"].dt.to_period("M"),
)
holdout_labels = holdout_outcomes["is_late"]
holdout_monthly_ranking = score_by_month(holdout_outcomes, holdout_probabilities)

holdout_ranking_report = holdout_monthly_ranking.copy()
holdout_ranking_report.index = holdout_ranking_report.index.astype(str)
holdout_ranking_report.loc["Overall"] = {
    "orders": len(holdout_labels),
    "late_orders": holdout_labels.sum(),
    "late_rate": holdout_labels.mean(),
    "average_precision": average_precision_score(holdout_labels, holdout_probabilities),
    "roc_auc": roc_auc_score(holdout_labels, holdout_probabilities),
}
holdout_ranking_report.round(4)

,orders,late_orders,late_rate,average_precision,roc_auc
approval_month,,,,,
2018-06,6094,71,0.0117,0.0707,0.7941
2018-07,6021,193,0.0321,0.0925,0.6438
2018-08,6495,406,0.0625,0.0859,0.5992
Overall,18610,670,0.0360,0.0799,0.7020


In [46]:
holdout_review = score_review_queue(holdout_outcomes, holdout_probabilities, review_fraction)
holdout_review["expected_random_late"] = (
    holdout_review["reviewed_orders"] * holdout_monthly_ranking["late_rate"]
)

review_count = holdout_review["reviewed_orders"].sum()
late_found = holdout_review["late_found"].sum()
total_late = holdout_review["total_late"].sum()

holdout_review_report = holdout_review.copy()
holdout_review_report.index = holdout_review_report.index.astype(str)
holdout_review_report.loc["Overall"] = {
    "reviewed_orders": review_count,
    "late_found": late_found,
    "total_late": total_late,
    "precision": late_found / review_count,
    "recall": late_found / total_late,
    "expected_random_late": holdout_review["expected_random_late"].sum(),
}
holdout_review_report.round(4)

,reviewed_orders,late_found,total_late,precision,recall,expected_random_late
approval_month,,,,,,
2018-06,610,25,71,0.0410,0.3521,7.1070
2018-07,603,47,193,0.0779,0.2435,19.3288
2018-08,650,59,406,0.0908,0.1453,40.6313
Overall,1863,131,670,0.0703,0.1955,67.0671


### Holdout findings

The selected model achieved an overall AP of 0.0799
and ROC-AUC of 0.7020. The holdout late rate was
3.60%.

The monthly review queues contained 1,863 orders
and found 131 of the
670 late deliveries. Overall queue precision was
7.0% and recall was 19.6%.
Random review at the same monthly volumes would find about
67 late orders.

Recall fell from 35.2% in June to 14.5% in August. The review queue simulates a fixed
monthly workload using each month's full set of scores.